### 04_pose_clustering

Creates a embedding map csv with pose and simple PAD (presentation attack detection) annotation for pose clustering
* Detects face and landmarks
* Computes poses using yaw/pitch/roll:
    * Yaw: left/right turn of head
        * left = negative yaw
        * right = positive yaw
    * Pitch: up/down movement of head 
        * up = negative pitch
        * down = positive pitch 
    * Roll: tilting left/right of head
        * left = negative roll
        * right = positive roll
    * Possible poses include: front, up, down, left, right
* Computes "liveness" score 
    * Allows for filtering low-quality images, printed images, images with unrealistic colorings, etc (mainly to reduce noise)
* Creates checkpoints during face and pose detection
* Use Kmeans to cluster dataset by poses (useful for pose-aware template building later)

* Uses: `scripts/pose.py` and `scripts/pad.py`
* INPUT: `data_processed/vggface2/enroll/embeddings/embeddings_map.csv`
* OUT: 
    * Checkpoints saved in: `data_processed/<dataset>/embeddings/checkpoints`
    * Annotated map saved in : `data_processed/vggface2/enroll/embeddings/` as `embeddings_map_with_pose.csv`

In [1]:
# ----- Imports and config -----
import time
import os, sys
import torch
from pathlib import Path
import pandas as pd
from facenet_pytorch import MTCNN
from sklearn.cluster import KMeans
from PIL import Image
import cv2
import numpy as np
from tqdm.auto import tqdm
import importlib

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# If scripts have been modified, uncomment the two lines below and reload them
# import scripts
# importlib.reload(scripts.pose)

# local helpers
from scripts.pose import landmarks_to_pose
from scripts.pad import heuristic_liveness_score

In [31]:
# ----- Helper functions -----

# KMeans clustering on yaw to assign pose clusters
def k_means_pose_clustering(df, MAP_PATH, K=3):
    # CLUSTER: KMeans on yaw (default K=3) or use deterministic bins
    USE_KMEANS = True
    if USE_KMEANS:
        X = df['yaw'].fillna(0.0).to_numpy().reshape(-1,1).astype(float)
        k = min(K, max(1, X.shape[0]))
        if k == 1:
            df['pose_cluster'] = 0
        else:
            km = KMeans(n_clusters=k, random_state=12345)
            labels = km.fit_predict(X)
            df['pose_cluster'] = labels.astype(int)
    else:
        bins = [-999, -15.0, 15.0, 999]
        labels = pd.cut(df['yaw'].fillna(0.0), bins=bins, labels=False)
        df['pose_cluster'] = labels.fillna(1).astype(int)

    # write augmented map
    OUT_MAP = MAP_PATH.with_name(MAP_PATH.stem + f"_with_pose_k{K}.csv")
    df.to_csv(OUT_MAP, index=False)

    # final minimal summaries
    print(f"Wrote: {OUT_MAP}")
    return df

# Diagnostic: show summary of yaw, per-cluster liveness
def summarize_pose_clusters(liveness_file, out_person_path=None, out_summary_path=None):
    liveness_file = Path(liveness_file)
    df = pd.read_csv(liveness_file)
    # cluster -> mean yaw (helps assign left/frontal/right)
    centers = df.groupby('pose_cluster')['yaw'].agg(['mean','std','count']).sort_values('mean')
    print("Cluster yaw summary:\n", centers)
    # per-cluster liveness
    print("Per-cluster liveness:\n", df.groupby('pose_cluster')['liveness_score'].agg(['mean','std','count']))

    order = centers.reset_index().sort_values('mean')['pose_cluster'].tolist()
    label_map = {}
    labels = ['left','frontal','right'] if len(order)==3 else [f'c{i}' for i in order]
    for lbl, cid in zip(labels, order):
        label_map[cid] = lbl
    print("Label map:", label_map)
    df['pose_label'] = df['pose_cluster'].map(label_map)

    print("\nOverall liveness mean,std:", df['liveness_score'].mean(), df['liveness_score'].std())
    for cid, g in df.groupby('pose_cluster'):
        print(f"Cluster {cid} ({label_map.get(cid,cid)}): n={len(g)}, liveness_mean={g['liveness_score'].mean():.3f}, below0.5={ (g['liveness_score']<0.5).mean():.3f}")

    print("\n")
    # compute mean liveness per person (and optionally save)
    person_live = df.groupby('person_id')['liveness_score'].agg(['mean','count']).reset_index().rename(columns={'mean':'mean_liveness','count':'n_images'})
    # write files if paths provided
    if out_person_path is not None:
        Path(out_person_path).parent.mkdir(parents=True, exist_ok=True)
        person_live.to_csv(out_person_path, index=False)
        print("Wrote person-level liveness:", out_person_path)

    if out_summary_path is not None:
        Path(out_summary_path).parent.mkdir(parents=True, exist_ok=True)
        centers.to_csv(out_summary_path)
        print("Wrote cluster summary:", out_summary_path)
        
    print(person_live.describe())

    print("\nDataFrame summary:")
    print("rows:", len(df))
    print("columns:", df.columns.tolist())
    print("pose_cluster unique:", df.get('pose_cluster').unique() if 'pose_cluster' in df.columns else 'no column')
    print("yaw NaNs:", df['yaw'].isna().sum(), "non-NaN:", df['yaw'].notna().sum())
    print("yaw describe:\n", df['yaw'].describe())
    # show a few yaw values
    print(df['yaw'].dropna().head(20).to_list()[:20])

    return df, person_live, centers

# Quick simulation: Simulate candidate K vs MIN_IMAGES thresholds
def simulate_k_values(path):
    
    df = pd.read_csv(path)

    valid = df['yaw'].notna()
    yaw = df.loc[valid, 'yaw'].to_numpy().reshape(-1,1).astype(float)

    candidate_K = [2, 3, 4, 5, 7]
    min_images_list = [1, 2, 3, 5]

    rows = []
    for K in candidate_K:
        # cluster only valid rows
        if yaw.shape[0] == 0:
            labels_full = np.full(len(df), -1, dtype=int)  # all unknown
        else:
            k = min(K, max(1, yaw.shape[0]))
            if k == 1:
                labels = np.zeros(yaw.shape[0], dtype=int)
            else:
                km = KMeans(n_clusters=k, random_state=12345)
                labels = km.fit_predict(yaw)

            # map labels back into full-length array; invalid yaw -> -1
            labels_full = np.full(len(df), -1, dtype=int)
            labels_full[valid.to_numpy()] = labels

        tmp = df.copy()
        tmp['sim_label'] = labels_full

        # compute centers ignoring unknowns
        valid_tmp = tmp[tmp['sim_label'] >= 0]
        if valid_tmp.shape[0] == 0:
            centers = pd.Series(dtype=float)
        else:
            centers = valid_tmp.groupby('sim_label')['yaw'].mean().sort_values()

        order = centers.index.tolist()
        mapping = {old: new for new, old in enumerate(order)}
        # remap only valid labels
        tmp.loc[tmp['sim_label'] >= 0, 'sim_label'] = tmp.loc[tmp['sim_label'] >= 0, 'sim_label'].map(mapping).astype(int)

        # grouped counts only for valid labels
        grouped = tmp[tmp['sim_label'] >= 0].groupby(['person_id', 'sim_label']).size().reset_index(name='count')

        for min_images in min_images_list:
            n_templates = int((grouped['count'] >= min_images).sum())
            # use agg on the Series to avoid the FutureWarning
            avg_tpls_per_person = grouped.groupby('person_id')['count'].agg(lambda s: (s >= min_images).sum()).mean()
            rows.append({
                'K': K,
                'min_images': min_images,
                'n_templates': n_templates,
                'avg_templates_per_person': float(avg_tpls_per_person),
                'frac_persons_with_>=3_poses': float((tmp[tmp['sim_label'] >= 0].groupby('person_id')['sim_label'].nunique() >= 3).mean())
            })

    res = pd.DataFrame(rows)
    print("Templates by K / min_images:\n", res.pivot(index='K', columns='min_images', values='n_templates'))
    print("\nAvg templates per person (sample):\n", res.pivot(index='K', columns='min_images', values='avg_templates_per_person'))
    print("\nFraction persons with >=3 pose clusters (same for all min_images):")
    print(res[['K','frac_persons_with_>=3_poses']].drop_duplicates().set_index('K'))

    # per-person coverage histogram for this K
    hist = tmp[tmp['yaw'].notna()].groupby('person_id')['sim_label'].nunique().value_counts().sort_index()
    print(f"K={K} per-person pose-coverage (num persons -> count):\n", hist)
    return res, hist


In [ ]:
# # POSE + PAD annotation (with alignment checks & checkpointing)
# def annotate_pose_and_liveness(
#         SPLIT,
#         MAP_PATH,
#         CHECKPOINT_DIR,
#         LOG_EVERY=5000,
#         MIN_FACE_SIDE=80,
#         MIN_CONF=0.9,
#         CHECKPOINT_EVERY=1000,
# ):
    
#     IMG_ROOT = None   # optional prefix
#     DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#     # checkpoint directory (will be created if missing)
#     CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

#     print(f"Annotating split='{SPLIT}' device={DEVICE}")

#     # load map and paths
#     df = pd.read_csv(MAP_PATH)
#     print(f"  Loaded map: {MAP_PATH} with {len(df)} entries")
#     if IMG_ROOT:
#         paths = [str(Path(IMG_ROOT) / p) for p in df['image_path'].astype(str).tolist()]
#     else:
#         paths = df['image_path'].astype(str).tolist()
#     total = len(paths)
#     print(f"  Images to process: {total}")

#     # setup MTCNN
#     mtcnn = MTCNN(keep_all=False, device=DEVICE)

#     # output containers
#     yaws, pitches, rolls = [], [], []
#     landmark_list = []
#     liveness_scores = []
#     aligned_flags = []
#     box_areas = []

#     # iterate once: compute landmarks -> pose and compute heuristic liveness from loaded image
#     for i, p in enumerate(tqdm(paths, desc="Annotating", total=total), start=1):
#         try:
#             # robust resolution: try as-is, then repo-root, then cwd
#             p_path = Path(p)
#             if not p_path.exists():
#                 p_path = PROJECT_ROOT / p_path
#             if not p_path.exists():
#                 p_path = Path.cwd() / p_path
#             if not p_path.exists():
#                 # couldn't find file -> record NaNs and continue
#                 yaws.append(float("nan")); pitches.append(float("nan")); rolls.append(float("nan"))
#                 landmark_list.append(None)
#                 liveness_scores.append(float("nan"))
#                 aligned_flags.append(False)
#                 box_areas.append(0.0)
#                 continue
            
#             img = Image.open(p_path).convert("RGB")
#             w, h = img.size
#             box, probs, landmarks = mtcnn.detect(img, landmarks=True)
#             if landmarks is None or len(landmarks) == 0:
#                 yaws.append(float("nan")); pitches.append(float("nan")); rolls.append(float("nan"))
#                 landmark_list.append(None)
#                 aligned_flags.append(False)
#                 box_areas.append(0.0)
#             else:
#                 lm = landmarks[0]  # (5,2)
#                 yaw, pitch, roll = landmarks_to_pose(lm, (w, h))
#                 yaws.append(yaw); pitches.append(pitch); rolls.append(roll)
#                 landmark_list.append(lm.tolist())

#                 # compute bounding box area & basic aligned-flag
#                 if box is not None and len(box) > 0:
#                     b = box[0]
#                     bw = max(0.0, float(b[2] - b[0])); bh = max(0.0, float(b[3] - b[1]))
#                     box_area = bw * bh
#                     box_areas.append(box_area)
#                     conf = float(probs[0]) if (probs is not None and len(probs) > 0) else 0.0
#                     aligned = (min(bw, bh) >= MIN_FACE_SIDE) and (conf >= MIN_CONF)
#                     aligned_flags.append(bool(aligned))
#                 else:
#                     box_areas.append(0.0)
#                     aligned_flags.append(False)

#             # compute liveness from the same loaded image (avoid second file read)
#             bgr = cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2BGR)
#             live = heuristic_liveness_score(bgr)
#             liveness_scores.append(float(live))
#         except Exception:
#             yaws.append(float("nan")); pitches.append(float("nan")); rolls.append(float("nan"))
#             landmark_list.append(None)
#             liveness_scores.append(float("nan"))
#             aligned_flags.append(False)
#             box_areas.append(0.0)

#         # periodic lightweight logging
#         if (i % LOG_EVERY) == 0:
#             print(f"  processed {i}/{total} images (last path: {Path(p).name})")

#         # checkpoint: write partial CSV so long runs can be resumed/inspected
#         if (i % CHECKPOINT_EVERY) == 0:
#             ck = pd.DataFrame({
#                 "image_path": paths[:i],
#                 "yaw": yaws,
#                 "pitch": pitches,
#                 "roll": rolls,
#                 "pose_landmarks": landmark_list,
#                 "liveness_score": liveness_scores,
#                 "face_aligned": aligned_flags,
#                 "face_box_area": box_areas,
#             })
#             ts = time.strftime("%Y%m%dT%H%M%S")
#             fname = f"embeddings_map_with_pose_checkpoint_{i}_{ts}.csv"
#             tmpf = CHECKPOINT_DIR / (fname + ".tmp")
#             finalf = CHECKPOINT_DIR / fname
#             # atomic-ish write
#             ck.to_csv(tmpf, index=False)
#             os.replace(str(tmpf), str(finalf))



#     # attach columns to full dataframe
#     df['yaw'] = yaws
#     df['pitch'] = pitches
#     df['roll'] = rolls
#     df['pose_landmarks'] = landmark_list
#     df['liveness_score'] = liveness_scores
#     df['face_aligned'] = aligned_flags
#     df['face_box_area'] = box_areas

#     return df, MAP_PATH


In [ ]:
from scripts.liveness import LiveFacePipelineFull

# POSE + PAD annotation (with alignment checks & external checkpointing)
def annotate_pose_and_liveness(
        SPLIT,
        MAP_PATH,
        CHECKPOINT_DIR,
        LOG_EVERY=5000,
        MIN_FACE_SIDE=80,
        MIN_CONF=0.9,
        CHECKPOINT_EVERY=1000,
):
    """
    Annotate a split with:
      - yaw, pitch, roll
      - pose_landmarks (5 points)
      - liveness_score
      - face_aligned
      - face_box_area

    Uses LiveFacePipelineFull for per-image processing and handles
    checkpointing at the notebook level.
    """

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Annotating split='{SPLIT}' device={DEVICE}")

    # Load map
    df = pd.read_csv(MAP_PATH)
    print(f"  Loaded map: {MAP_PATH} with {len(df)} entries")

    # Resolve image paths (same logic as before, but simpler)
    paths = df["image_path"].astype(str).tolist()
    total = len(paths)
    print(f"  Images to process: {total}")

    # Initialize unified pipeline
    pipe = LiveFacePipelineFull(
        device=DEVICE,
        min_face_side=MIN_FACE_SIDE,
        min_conf=MIN_CONF,
        image_size=160,
    )

    # Storage for results (per image)
    results = []

    for i, p in enumerate(tqdm(paths, desc="Annotating", total=total), start=1):
        # robust path resolution (similar to old code)
        try:
            p_path = Path(p)
            if not p_path.exists():
                p_path = PROJECT_ROOT / p_path
            if not p_path.exists():
                p_path = Path.cwd() / p_path

            if not p_path.exists():
                # could not resolve file -> record empty result
                info = {
                    "yaw": float("nan"),
                    "pitch": float("nan"),
                    "roll": float("nan"),
                    "landmarks": None,
                    "liveness_score": float("nan"),
                    "face_aligned": False,
                    "face_box_area": 0.0,
                    "embedding": None,
                }
            else:
                info = pipe.process_image(p_path)
        except Exception:
            # in case of any unexpected error, record empty result
            info = {
                "yaw": float("nan"),
                "pitch": float("nan"),
                "roll": float("nan"),
                "landmarks": None,
                "liveness_score": float("nan"),
                "face_aligned": False,
                "face_box_area": 0.0,
                "embedding": None,
            }

        results.append(info)

        # periodic lightweight logging
        if (i % LOG_EVERY) == 0:
            print(f"  processed {i}/{total} images (last path: {Path(p).name})")

        # checkpoint: write partial CSV so long runs can be resumed/inspected
        if (i % CHECKPOINT_EVERY) == 0:
            ck_df = df.iloc[:i].copy()
            ck_df["yaw"]             = [r["yaw"] for r in results]
            ck_df["pitch"]           = [r["pitch"] for r in results]
            ck_df["roll"]            = [r["roll"] for r in results]
            ck_df["pose_landmarks"]  = [r["landmarks"] for r in results]
            ck_df["liveness_score"]  = [r["liveness_score"] for r in results]
            ck_df["face_aligned"]    = [r["face_aligned"] for r in results]
            ck_df["face_box_area"]   = [r["face_box_area"] for r in results]

            ts = time.strftime("%Y%m%dT%H%M%S")
            fname = f"embeddings_map_with_pose_checkpoint_{i}_{ts}.csv"
            tmpf = CHECKPOINT_DIR / (fname + ".tmp")
            finalf = CHECKPOINT_DIR / fname

            ck_df.to_csv(tmpf, index=False)
            os.replace(str(tmpf), str(finalf))
            print(f"  [checkpoint] wrote {finalf}")

    # Attach columns to full dataframe
    df["yaw"]             = [r["yaw"] for r in results]
    df["pitch"]           = [r["pitch"] for r in results]
    df["roll"]            = [r["roll"] for r in results]
    df["pose_landmarks"]  = [r["landmarks"] for r in results]
    df["liveness_score"]  = [r["liveness_score"] for r in results]
    df["face_aligned"]    = [r["face_aligned"] for r in results]
    df["face_box_area"]   = [r["face_box_area"] for r in results]

    return df, MAP_PATH


In [ ]:
# ----- Main processing -----
OUT_DIR = Path("../data_processed/vggface2")
SPLIT = "enroll"   
MAP_PATH = OUT_DIR / "embeddings" / SPLIT / "embeddings_map.csv"
CHECKPOINT_DIR = OUT_DIR / "embeddings" / SPLIT / "checkpoints"

# Progress + heuristics (adjust as needed)
LOG_EVERY = 5000            # print a short line every LOG_EVERY images
MIN_FACE_SIDE = 80          # min face side (px) for aligned face
MIN_CONF = 0.9              # min face detection confidence for aligned face
CHECKPOINT_EVERY = 2000     # write a checkpoint CSV every N images

MAP_FILE = OUT_DIR / "embeddings" / SPLIT / "embeddings_map_with_pose.csv"

# Controls
FORCE_RERUN = False               # set True to always recompute
MIN_VALID_YAW_FRAC = 0.20         # at least 20% rows must have valid yaw to consider file complete

need_annotate = False
if not MAP_FILE.exists():
    need_annotate = True
else:
    df = pd.read_csv(MAP_FILE)
    if FORCE_RERUN:
        need_annotate = True
    else:
        # check that pose columns exist and are sufficiently populated
        if 'yaw' not in df.columns or 'pose_landmarks' not in df.columns:
            need_annotate = True
        else:
            valid_frac = df['yaw'].notna().mean()
            if valid_frac < MIN_VALID_YAW_FRAC:
                need_annotate = True

if need_annotate:
    print(f"Annotating (map missing or incomplete). Writing to: {MAP_FILE}")
    df, MAP_PATH = annotate_pose_and_liveness(
        SPLIT,
        MAP_PATH,
        CHECKPOINT_DIR,
        LOG_EVERY=LOG_EVERY,
        MIN_FACE_SIDE=MIN_FACE_SIDE,
        MIN_CONF=MIN_CONF,
        CHECKPOINT_EVERY=CHECKPOINT_EVERY,
    )
    df = k_means_pose_clustering(df, MAP_PATH, K=3)
else:
    print(f"Loading existing map with pose/liveness: {MAP_FILE}")

print("  pose_cluster counts:", df['pose_cluster'].value_counts(dropna=False).to_dict())
print("  liveness (mean,std):", float(df['liveness_score'].mean(skipna=True)), float(df['liveness_score'].std(skipna=True)))


Annotating split='enroll' device=cuda
  Loaded map: ..\data_processed\vggface2\embeddings\enroll\embeddings_map.csv with 140922 entries
  Images to process: 140922


Annotating:   0%|          | 0/140922 [00:00<?, ?it/s]

  processed 5000/140922 images (last path: 0042_01.jpg)
  processed 10000/140922 images (last path: 0176_01.jpg)
  processed 15000/140922 images (last path: 0138_02.jpg)
  processed 20000/140922 images (last path: 0254_01.jpg)
  processed 25000/140922 images (last path: 0107_02.jpg)
  processed 30000/140922 images (last path: 0220_01.jpg)
  processed 35000/140922 images (last path: 0094_03.jpg)
  processed 40000/140922 images (last path: 0030_01.jpg)
  processed 45000/140922 images (last path: 0227_01.jpg)
  processed 50000/140922 images (last path: 0034_01.jpg)
  processed 55000/140922 images (last path: 0257_01.jpg)
  processed 60000/140922 images (last path: 0319_01.jpg)
  processed 65000/140922 images (last path: 0380_01.jpg)
  processed 70000/140922 images (last path: 0022_01.jpg)
  processed 75000/140922 images (last path: 0048_01.jpg)
  processed 80000/140922 images (last path: 0224_01.jpg)
  processed 85000/140922 images (last path: 0197_01.jpg)
  processed 90000/140922 images 

In [34]:
# Print summary of pose clusters + liveness
OUT_DIR = Path("../data_processed/vggface2/enroll_cluster/")
liveness_file = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_k3.csv")

k=3
out_person = OUT_DIR / f"person_mean_liveness_k{k}.csv"
out_centers = OUT_DIR / f"pose_cluster_centers_k{k}.csv"
df_loaded, person_live, centers = summarize_pose_clusters(liveness_file, out_person_path=out_person, out_summary_path=out_centers)


Cluster yaw summary:
                    mean        std  count
pose_cluster                             
0            -39.525904   8.719568  64727
2            -10.143954  10.449657  33840
1             45.385105  10.661086  42134
Per-cluster liveness:
                   mean       std  count
pose_cluster                           
0             0.582069  0.092352  64727
1             0.584140  0.090764  42134
2             0.579990  0.084379  34061
Label map: {0: 'left', 2: 'frontal', 1: 'right'}

Overall liveness mean,std: 0.5821858707751474 0.09002013938235794
Cluster 0 (left): n=64727, liveness_mean=0.582, below0.5=0.125
Cluster 1 (right): n=42134, liveness_mean=0.584, below0.5=0.121
Cluster 2 (frontal): n=34061, liveness_mean=0.580, below0.5=0.120


Wrote person-level liveness: ..\data_processed\vggface2\enroll_cluster\person_mean_liveness_k3.csv
Wrote cluster summary: ..\data_processed\vggface2\enroll_cluster\pose_cluster_centers_k3.csv
       mean_liveness    n_images
count    

In [35]:
# Determine if larger K is possible

path = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_k3.csv")

# simulate various k values to see if larger K is possible
res, hist = simulate_k_values(path)

Templates by K / min_images:
 min_images     1     2     3     5
K                                 
2            960   960   960   960
3           1440  1440  1440  1440
4           1920  1920  1920  1916
5           2400  2400  2400  2396
7           3357  3348  3327  3246

Avg templates per person (sample):
 min_images        1      2        3         5
K                                            
2           2.00000  2.000  2.00000  2.000000
3           3.00000  3.000  3.00000  3.000000
4           4.00000  4.000  4.00000  3.991667
5           5.00000  5.000  5.00000  4.991667
7           6.99375  6.975  6.93125  6.762500

Fraction persons with >=3 pose clusters (same for all min_images):
   frac_persons_with_>=3_poses
K                             
2                          0.0
3                          1.0
4                          1.0
5                          1.0
7                          1.0
K=7 per-person pose-coverage (num persons -> count):
 sim_label
6      3
7    477

In [36]:
# K=5 clustering and summary

df = k_means_pose_clustering(df, MAP_PATH, K=5)

OUT_DIR = Path("../data_processed/vggface2/enroll/")
liveness_file = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_k5.csv")

k=5
out_person = OUT_DIR / f"person_mean_liveness_k{k}.csv"
out_centers = OUT_DIR / f"pose_cluster_centers_k{k}.csv"
df_loaded, person_live, centers = summarize_pose_clusters(liveness_file, out_person_path=out_person, out_summary_path=out_centers)

Wrote: ..\data_processed\vggface2\embeddings\enroll\embeddings_map_with_pose_k5.csv
Cluster yaw summary:
                    mean       std  count
pose_cluster                            
3            -44.032063  6.937113  43207
0            -26.731986  5.701574  34107
2             -4.548936  8.011246  20820
1             35.974556  7.221712  19236
4             52.613899  7.220225  23331
Per-cluster liveness:
                   mean       std  count
pose_cluster                           
0             0.580021  0.087727  34107
1             0.580965  0.088530  19236
2             0.580266  0.084026  21041
3             0.582902  0.093864  43207
4             0.586764  0.092298  23331
Label map: {3: 'c3', 0: 'c0', 2: 'c2', 1: 'c1', 4: 'c4'}

Overall liveness mean,std: 0.5821858707751474 0.09002013938235794
Cluster 0 (c0): n=34107, liveness_mean=0.580, below0.5=0.121
Cluster 1 (c1): n=19236, liveness_mean=0.581, below0.5=0.124
Cluster 2 (c2): n=21041, liveness_mean=0.580, below0.5=0.1

In [39]:
# Basic stats on annotated data
p = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_k5.csv")
df = pd.read_csv(p, usecols=['embedding_index','image_path','yaw','pose_landmarks','liveness_score','face_aligned','face_box_area'])

total = len(df)
na_landmarks = df['pose_landmarks'].isna().sum()
small_box = (df['face_box_area'] <= 0).sum()   # area==0 indicates missing box
# also compute inferred min-side threshold (approx) if you stored bw/bh; else estimate sqrt(area)
min_side_est = (df['face_box_area']**0.5)  # rough estimate; compare with MIN_FACE_SIDE
small_side_count = (min_side_est < 80).sum()  # same threshold as notebook default
print("total:", total)
print("missing landmarks:", na_landmarks)
print("zero box area (likely missing):", small_box)
print("estimated min-side < 80 (approx):", small_side_count)
print("face_aligned True:", df['face_aligned'].sum(), "False:", total - df['face_aligned'].sum())

# distribution of yaw for aligned vs not-aligned
print("yaw describe for aligned rows:")
print(df[df['face_aligned']==True]['yaw'].describe())
print("yaw describe for not-aligned rows:")
print(df[df['face_aligned']==False]['yaw'].describe())

# liveness distribution vs alignment
print("mean liveness aligned:", df[df['face_aligned']==True]['liveness_score'].mean())
print("mean liveness not-aligned:", df[df['face_aligned']==False]['liveness_score'].mean())

# approximate min-side from area
df['min_side_est'] = (df['face_box_area']**0.5)
print("min_side_est describe:\n", df['min_side_est'].describe())

# correlation between box area and liveness
print("corr(box_area, liveness):", df['face_box_area'].corr(df['liveness_score']))

# fraction of images below thresholds
print("frac min-side <80:", (df['min_side_est'] < 80).mean())
print("face_aligned True fraction:", df['face_aligned'].mean())

total: 140922
missing landmarks: 221
zero box area (likely missing): 221
estimated min-side < 80 (approx): 50532
face_aligned True: 77246 False: 63676
yaw describe for aligned rows:
count    77246.000000
mean        -7.100773
std         36.608268
min        -83.574682
25%        -37.847841
50%        -20.421268
75%         34.674390
max         88.974482
Name: yaw, dtype: float64
yaw describe for not-aligned rows:
count    63455.000000
mean        -6.948267
std         38.553954
min        -84.433638
25%        -38.581255
50%        -21.567141
75%         37.233534
max         89.081415
Name: yaw, dtype: float64
mean liveness aligned: 0.5646933111162381
mean liveness not-aligned: 0.6034062719217667
min_side_est describe:
 count    140922.000000
mean        122.990099
std          81.919153
min           0.000000
25%          66.211589
50%         100.156241
75%         156.819664
max        2017.630712
Name: min_side_est, dtype: float64
corr(box_area, liveness): -0.2158248976364193
fr